# Learning in games: live demonstrations

Companion to [Lecture 6](https://tomaskroupa.github.io/quarto-game-theory-course/lectures/06-learning-in-games.html).
Select a **Python 3** kernel in VS Code or Colab and **Run All**. Then use the controls below each method;
sliders redraw the figures when released. The notebook is self-contained.

In [10]:
from io import BytesIO
import numpy as np
from typing import TypedDict
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = ("#155a8a", "#d97924")

# Explicit dictionary fields keep VS Code's type checks precise.
class Game(TypedDict):
    A: np.ndarray
    B: np.ndarray
    labels: tuple[str, ...]
    P: np.ndarray | None
    NE: list[tuple[float, float]]


class History(TypedDict):
    actions: np.ndarray
    t: np.ndarray
    freq: np.ndarray
    joint: np.ndarray
    payoffs: np.ndarray
    regrets: np.ndarray
    rho: np.ndarray
    nash_error: np.ndarray
    gap: np.ndarray | None


# All payoff tables also appear in the textbook (_variables.yml).
# A[i, j] = Alice's payoff; B[i, j] = Bob's payoff.
q_A, q_B = np.meshgrid(np.arange(7.), np.arange(7.), indexing="ij")
MORRA = np.array([
    [0, 2, 2, -3, 0, 0, -4, 0, 0],
    [-2, 0, 0, 0, 3, 3, -4, 0, 0],
    [-2, 0, 0, -3, 0, 0, 0, 4, 4],
    [3, 0, 3, 0, -4, 0, 0, -5, 0],
    [0, -3, 0, 4, 0, 4, 0, -5, 0],
    [0, -3, 0, 0, -4, 0, 5, 0, 5],
    [4, 4, 0, 0, 0, -5, 0, 0, -6],
    [0, 0, -4, 5, 5, 0, 0, 0, -6],
    [0, 0, -4, 0, 0, -5, 6, 6, 0]
], dtype=float)
GAMES: dict[str, Game] = {
    "Coordination": {
        "A": np.array([[2., 0.], [0., 1.]]),
        "B": np.array([[2., 0.], [0., 1.]]),
        "labels": ("a", "b"), "P": np.array([[2., 0.], [0., 1.]]),
        "NE": [(1., 1.), (0., 0.), (1/3, 1/3)]},
    "Two-route congestion": {
        "A": np.array([[-3., -1.], [-2., -2.]]),
        "B": np.array([[-3., -2.], [-1., -2.]]),
        "labels": ("e1", "e2"), "P": np.array([[-4., -3.], [-3., -4.]]),
        "NE": [(1., 0.), (0., 1.), (.5, .5)]},
    "Matching pennies": {
        "A": np.array([[1., -1.], [-1., 1.]]),
        "B": np.array([[-1., 1.], [1., -1.]]),
        "labels": ("H", "T"), "P": None, "NE": [(.5, .5)]},
    "Prisoner's dilemma": {
        "A": np.array([[-1., -4.], [0., -3.]]),
        "B": np.array([[-1., 0.], [-4., -3.]]),
        "labels": ("C", "D"), "P": np.array([[0., 1.], [1., 2.]]),
        "NE": [(0., 0.)]},
    "Bach or Stravinski": {
        "A": np.array([[2., 0.], [0., 1.]]),
        "B": np.array([[1., 0.], [0., 2.]]),
        "labels": ("B", "S"), "P": np.array([[2., 1.], [0., 2.]]),
        "NE": [(1., 1.), (0., 0.), (2/3, 1/3)]},
    "Cournot duopoly (7 quantities)": {
        "A": q_A * (12 - q_A - q_B),
        "B": q_B * (12 - q_A - q_B),
        "labels": tuple(str(i) for i in range(7)),
        "P": 12*(q_A + q_B) - q_A**2 - q_B**2 - q_A*q_B, "NE": []},
    "Morra (9 strategies)": {
        "A": MORRA, "B": -MORRA,
        "labels": tuple(f"{i}-{j}" for i in range(1, 4) for j in range(1, 4)),
        "P": None, "NE": []},
}

tables = []
for name, g in GAMES.items():
    labels, A, B = g["labels"], g["A"], g["B"]
    k = len(labels)
    rows = "".join("<tr><th>" + labels[i] + "</th>" + "".join(
        f"<td style='padding:5px 9px'>{A[i,j]:g}, {B[i,j]:g}</td>"
        for j in range(k)) + "</tr>" for i in range(k))
    tables.append(f"<details style='margin:10px'><summary><b>{name}</b> ({k} x {k})</summary>"
                  + "<table><tr><th></th>" + "".join(f"<th>{s}</th>" for s in labels)
                  + "</tr>" + rows + "</table></details>")
display(HTML("<p>Click a game to show its payoffs. Alice = rows; Bob = columns. "
             "Entries: Alice's payoff, Bob's payoff.</p>" + "".join(tables)))

,a,b
a,"2, 2","0, 0"
b,"0, 0","1, 1"
,e1,e2
e1,"-3, -3","-1, -2"
e2,"-2, -1","-2, -2"
,H,T
H,"1, -1","-1, 1"
T,"-1, 1","1, -1"
,C,D
C,"-1, -1","-4, 0"


In [11]:
def best_response(values, tie="First listed", rng=None):
    best = np.flatnonzero(np.isclose(values, np.max(values), rtol=0, atol=1e-12))
    if tie == "Random":
        if rng is None:
            rng = np.random.default_rng()
        return int(rng.choice(best))
    return int(best[-1] if tie == "Last listed" else best[0])


def pure_equilibria(game: Game):
    A, B = game["A"], game["B"]
    return np.argwhere(np.isclose(A, A.max(axis=0)) &
                       np.isclose(B, B.max(axis=1)[:, None]))


def history_data(game: Game, actions) -> History:
    """Post-round diagnostics, recomputed from the realized action history."""
    A, B = game["A"], game["B"]
    k = len(game["labels"])
    actions = np.asarray(actions, dtype=int)
    t = np.arange(1, len(actions) + 1)
    a, b = actions.T
    freq = np.eye(k)[actions].cumsum(axis=0) / t[:, None, None]
    joint = np.eye(k*k)[k*a + b].cumsum(axis=0).reshape(-1, k, k) / t[:, None, None]
    payoffs = np.column_stack((A[a, b], B[a, b]))
    increments = np.stack((A[:, b].T - payoffs[:, 0, None],
                           B[a, :] - payoffs[:, 1, None]), axis=1)
    regrets = increments.cumsum(axis=0)
    rho = np.maximum(0, regrets.max(axis=2)) / t[:, None]
    x, y = freq[:, 0], freq[:, 1]
    row_values = np.einsum("ij,tj->ti", A, y)
    column_values = np.einsum("ti,ij->tj", x, B)
    dev_A = row_values.max(axis=1) - (x * row_values).sum(axis=1)
    dev_B = column_values.max(axis=1) - (y * column_values).sum(axis=1)
    nash_error = np.maximum(0, np.maximum(dev_A, dev_B))
    gap = (row_values.max(axis=1) - np.einsum("ti,ij->tj", x, A).min(axis=1)
           if np.allclose(A + B, 0) else None)
    return {"actions": actions, "t": t, "freq": freq, "joint": joint, "payoffs": payoffs,
            "regrets": regrets, "rho": rho, "nash_error": nash_error, "gap": gap}


def heatmap(ax, matrix, labels, title):
    k = len(labels)
    ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
    for i in range(k):
        for j in range(k):
            ax.text(j, i, f"{matrix[i,j]:.2f}" if k > 4 else f"{matrix[i,j]:.3f}",
                    ha="center", va="center", fontsize=7 if k > 4 else 10,
                    color="white" if matrix[i,j] > .6 else "#16191d")
    ax.set(xticks=range(k), yticks=range(k), xticklabels=labels, yticklabels=labels,
           xlabel="Bob", ylabel="Alice", title=title)
    ax.tick_params(labelsize=8 if k > 4 else 10)


def frequency_plot(ax, data: History, labels, player: int | None = None):
    if player is None:  # Two-action view: both players' first-action frequencies.
        for i, name in enumerate(("Alice", "Bob")):
            ax.plot(data["t"], data["freq"][:, i, 0], color=COLORS[i], label=name,
                    ls="--" if i else "-", marker="o" if len(data["t"]) == 1 else None)
        title, ylabel = "Empirical marginals (after each round)", f"Frequency of {labels[0]}"
    else:  # Larger games: show the whole distribution for one player.
        for a, label in enumerate(labels):
            ax.plot(data["t"], data["freq"][:, player, a], label=label,
                    color=plt.get_cmap("tab10")(a), marker="o" if len(data["t"]) == 1 else None)
        title, ylabel = f"{('Alice', 'Bob')[player]}: empirical frequencies", "Frequency"
    ax.set(xscale="log", ylim=(-.03, 1.03), xlabel="Rounds played, T", ylabel=ylabel, title=title)
    ax.grid(alpha=.2); ax.legend(fontsize=8, ncols=3 if player is not None else 1)


def error_plot(ax, data: History):
    marker = "o" if len(data["t"]) == 1 else None
    ax.plot(data["t"], data["nash_error"], color=COLORS[0], marker=marker, label="Maximum deviation incentive")
    gap = data["gap"]
    if gap is not None:
        ax.plot(data["t"], gap, color=COLORS[1], ls="--", marker=marker, label="Zero-sum saddle gap")
    ax.set(xscale="log", xlabel="Rounds played, T", ylabel="Utility units",
           title="Nash error of empirical marginals")
    ax.set_yscale("symlog", linthresh=1e-3); ax.set_ylim(bottom=0)
    ax.grid(alpha=.2); ax.legend(fontsize=8)


def distribution_plots(axes, data: History, labels):
    mu = data["joint"][-1]
    product = np.outer(*data["freq"][-1])
    heatmap(axes[0], mu, labels, "Empirical joint distribution")
    heatmap(axes[1], product, labels, "Product of empirical marginals")


def slider(value, minimum, maximum, description):
    return widgets.IntSlider(value=value, min=minimum, max=maximum, step=1,
                             description=description, continuous_update=False)


def dashboard(function, controls):
    for control in controls.values():
        control.style.description_width = "initial"
        control.layout.width = "290px"
    syncing = False

    def sync_actions(change=None):
        nonlocal syncing
        syncing = True
        try:
            labels = GAMES[controls["game"].value]["labels"]
            options = tuple((label, i) for i, label in enumerate(labels))
            for name in ("alice", "bob"):
                if name in controls and controls[name].options != options:
                    old = controls[name].value
                    controls[name].options = options
                    controls[name].value = min(old or 0, len(labels)-1)
        finally:
            syncing = False

    controls["game"].observe(sync_actions, names="value")
    sync_actions()

    # Replace one image in place; avoid accumulating inline plot outputs in VS Code.
    image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))

    def render(change=None):
        if syncing:
            return
        with plt.ioff():
            fig = function(**{name: control.value for name, control in controls.items()})
        try:
            with BytesIO() as buffer:
                fig.savefig(buffer, format="png", bbox_inches="tight")
                image.value = buffer.getvalue()
        finally:
            plt.close(fig)

    for control in controls.values():
        control.observe(render, names="value")
    render()
    display(widgets.VBox([widgets.HBox(list(controls.values()),
                                     layout=widgets.Layout(flex_flow="row wrap")), image]))
    return controls

## 1. Best-response dynamics

Start with **Cournot**, initial quantities **0, 0**, and Alice moving first. Each firm chooses 0–6 units;
price is 14 minus total output and unit cost is 2. Change the starting quantities and update order.
Compare sequential and simultaneous updates; use Matching pennies or Morra to follow a cycle.

In [12]:
def play_br(game: Game, start=(0, 1), steps=12, first=0, mode="Sequential"):
    A, B = game["A"], game["B"]
    actions, turn = [tuple(start)], first
    for _ in range(steps):
        a, b = actions[-1]
        values = (A[:, b], B[a, :])
        dissatisfied = [values[i][(a, b)[i]] < values[i].max() - 1e-12
                        for i in range(2)]
        if not any(dissatisfied):
            break
        new = [a, b]
        if mode == "Simultaneous":
            for i in range(2):
                if dissatisfied[i]:
                    new[i] = best_response(values[i])
        else:
            i = turn if dissatisfied[turn] else 1 - turn
            new[i] = best_response(values[i])
            turn = 1 - i
        actions.append(tuple(new))
    return np.asarray(actions)

In [13]:
def show_br(game="Cournot duopoly (7 quantities)", alice=0, bob=0, steps=12, first=0, mode="Sequential"):
    g = GAMES[game]
    A, B, labels = g["A"], g["B"], g["labels"]
    k = len(labels)
    actions = play_br(g, (alice, bob), steps, first, mode)
    n = len(actions)
    visits = np.bincount(k*actions[:, 0] + actions[:, 1], minlength=k*k).reshape(k, k)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), layout="constrained")
    ax = axes[0, 0]
    ax.imshow(visits, cmap="Blues", vmin=0, vmax=max(1, visits.max()*2))
    for i in range(k):
        for j in range(k):
            ax.text(j, i-.15, f"{A[i,j]:g}, {B[i,j]:g}", ha="center", fontsize=7 if k > 4 else 10)
            if visits[i, j]:
                ax.text(j, i+.28, f"visits: {visits[i,j]}", ha="center", fontsize=6 if k > 4 else 8)
    for i, j in pure_equilibria(g):
        ax.add_patch(Rectangle((j-.48, i-.48), .96, .96, fill=False, ec="#ad5270", lw=1.5))
    for old, new in dict.fromkeys(zip(map(tuple, actions[:-1]), map(tuple, actions[1:]))):
        ax.annotate("", xy=(new[1], new[0]), xytext=(old[1], old[0]),
                    arrowprops=dict(arrowstyle="->", color=COLORS[0], lw=2,
                                    shrinkA=10 if k > 4 else 25, shrinkB=10 if k > 4 else 25,
                                    connectionstyle="arc3,rad=.13"))
    ax.scatter(actions[0, 1], actions[0, 0], s=650*(2/k)**2, facecolors="none",
               edgecolors="#3b8b5e", lw=2, label="Start")
    ax.scatter(actions[-1, 1], actions[-1, 0], s=max(30, 160/k), color="#16191d", marker="x", label="Current")
    ax.plot([], [], color="#ad5270", label="Pure NE: outlined cell")
    ax.set(xticks=range(k), yticks=range(k), xticklabels=labels, yticklabels=labels,
           xlabel="Bob", ylabel="Alice", title="Payoffs and visited profiles")
    ax.tick_params(labelsize=8 if k > 4 else 10)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
    u = np.arange(n)
    for i, name in enumerate(("Alice", "Bob")):
        axes[0, 1].step(u, actions[:, i], where="post", marker="o", ms=3,
                        color=COLORS[i], ls="--" if i else "-", label=name)
        axes[1, 0].step(u, (A if i == 0 else B)[actions[:, 0], actions[:, 1]],
                        where="post", color=COLORS[i], ls="--" if i else "-", label=name)
    axes[0, 1].set(yticks=range(k), yticklabels=labels, ylim=(-.3, k-.7),
                   xlabel="Update (0 = initial profile)", title="Actions")
    axes[1, 0].set(xlabel="Update", ylabel="Utility", title="Payoffs at the current profile")
    axes[0, 1].legend(); axes[1, 0].legend()
    potential = g["P"]
    if potential is not None:
        axes[1, 1].step(u, potential[actions[:, 0], actions[:, 1]], where="post", marker="o", color="#3b8b5e")
        axes[1, 1].set(xlabel="Update", ylabel="Potential P", title="Potential along the path")
    else:
        a, b = actions.T
        gains = (A[:, b].max(axis=0)-A[a, b], B[a, :].max(axis=1)-B[a, b])
        for i, name in enumerate(("Alice", "Bob")):
            axes[1, 1].step(u, gains[i], where="post", color=COLORS[i], label=name,
                            ls="--" if i else "-")
        axes[1, 1].set(xlabel="Update", ylabel="Utility gain", title="Deviation incentives at current profile")
        axes[1, 1].legend()
    for ax in (axes[0, 1], axes[1, 0], axes[1, 1]):
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    a, b = actions[-1]
    equilibrium = A[a, b] == A[:, b].max() and B[a, b] == B[a, :].max()
    status = "Pure Nash equilibrium reached" if equilibrium else "Update limit reached"
    if not equilibrium and len(set(map(tuple, actions))) < n:
        status += "; a profile has been revisited"
    fig.suptitle(f"{game} | {mode} updates | Updates: {n-1} | {status}", fontsize=12)
    return fig


br_controls = dashboard(show_br, {
    "game": widgets.Dropdown(options=list(GAMES), value="Cournot duopoly (7 quantities)", description="Game"),
    "alice": widgets.Dropdown(options=[("0", 0)], description="Alice starts"),
    "bob": widgets.Dropdown(options=[("0", 0)], description="Bob starts"),
    "first": widgets.Dropdown(options=[("Alice", 0), ("Bob", 1)], description="First updater"),
    "mode": widgets.Dropdown(options=["Sequential", "Simultaneous"], description="Updates"),
    "steps": slider(12, 0, 60, "Update limit"),
})

def enable_first_updater(change=None):
    br_controls["first"].disabled = br_controls["mode"].value == "Simultaneous"

br_controls["mode"].observe(enable_first_updater, names="value")
enable_first_updater()

## 2. Fictitious play

The initial actions are **round 1**, with no prior counts. Compare Matching pennies, Cournot, and Morra.
Two-action games show the empirical probability path; larger games show all action frequencies for each player.

In [14]:
def play_fp(game: Game, rounds=500, start=(0, 0), tie="First listed", seed=1):
    A, B = game["A"], game["B"]
    k = len(game["labels"])
    rng = np.random.default_rng(seed)
    actions = [tuple(start)]
    counts = np.eye(k)[list(start)].copy()
    for t in range(1, rounds):
        x, y = counts / t
        # Both best responses use counts BEFORE either new action is added.
        a = best_response(A @ y, tie, rng)
        b = best_response(x @ B, tie, rng)
        actions.append((a, b))
        counts[0, a] += 1
        counts[1, b] += 1
    return np.asarray(actions)

In [15]:
def show_fp(game="Matching pennies", rounds=500, alice=0, bob=0, tie="First listed", seed=1):
    g = GAMES[game]
    data = history_data(g, play_fp(g, rounds, (alice, bob), tie, seed))
    labels, k = g["labels"], len(g["labels"])
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout="constrained")
    n = min(rounds, 80)
    for i, name in enumerate(("Alice", "Bob")):
        axes[0, 0].step(data["t"][:n], data["actions"][:n, i], where="post",
                        color=COLORS[i], ls="--" if i else "-", marker="o" if n == 1 else None, label=name)
    axes[0, 0].set(yticks=range(k), yticklabels=labels, ylim=(-.3, k-.7),
                   xlabel="Round", title=f"Actions: first {n} rounds")
    axes[0, 0].legend()
    if k == 2:
        frequency_plot(axes[0, 1], data, labels)
        x, y = data["freq"][:, 0, 0], data["freq"][:, 1, 0]
        axes[0, 2].plot(x, y, color=COLORS[0], alpha=.7)
        axes[0, 2].scatter(x[0], y[0], color="#3b8b5e", label="Start", zorder=3)
        axes[0, 2].scatter(x[-1], y[-1], color=COLORS[1], label="Current", zorder=3)
        ne = np.asarray(g["NE"])
        axes[0, 2].scatter(ne[:, 0], ne[:, 1], marker="x", s=65, color="black", label="Nash equilibria")
        axes[0, 2].set(xlim=(-.05, 1.05), ylim=(-.05, 1.05),
                       xlabel=f"Alice: empirical frequency of {labels[0]}",
                       ylabel=f"Bob: empirical frequency of {labels[0]}", title="Empirical probability path")
        axes[0, 2].legend(fontsize=8); axes[0, 2].grid(alpha=.2)
    else:
        frequency_plot(axes[0, 1], data, labels, player=0)
        frequency_plot(axes[0, 2], data, labels, player=1)
    error_plot(axes[1, 0], data)
    distribution_plots(axes[1, 1:], data, labels)
    fig.suptitle(f"{game} | Fictitious play | T = {rounds:,} | "
                 f"Nash error = {data['nash_error'][-1]:.4f}", fontsize=12)
    return fig


fp_controls = dashboard(show_fp, {
    "game": widgets.Dropdown(options=list(GAMES), value="Matching pennies", description="Game"),
    "rounds": slider(500, 1, 5000, "Rounds"),
    "alice": widgets.Dropdown(options=[("0", 0)], description="Alice starts"),
    "bob": widgets.Dropdown(options=[("0", 0)], description="Bob starts"),
    "tie": widgets.Dropdown(options=["First listed", "Last listed", "Random"], description="Ties choose"),
    "seed": slider(1, 0, 100, "Random seed"),
})

## 3. External regret matching

Change the seed and run length. Compare current sampling probabilities with empirical frequencies for every action.
Switch Bob to a fixed mixture to test Alice against a non-learning opponent; the probability left after the first
action is shared equally among his other actions. Only learning players carry a no-regret guarantee.

In [16]:
def regret_probabilities(regret: np.ndarray) -> np.ndarray:
    positive = np.maximum(regret, 0)
    return positive / positive.sum() if positive.sum() > 0 else np.full(len(regret), 1/len(regret))


def play_rm(game: Game, rounds=1000, seed=1, bob_rule="Regret matching", bob_first=.5):
    A, B = game["A"], game["B"]
    k = len(game["labels"])
    rng = np.random.default_rng(seed)
    regrets = np.zeros((2, k))
    actions, probabilities = [], []
    for _ in range(rounds):
        p = regret_probabilities(regrets[0])
        q = regret_probabilities(regrets[1])
        if bob_rule == "Fixed mixture":
            q = np.full(k, (1-bob_first)/(k-1))
            q[0] = bob_first
        a, b = rng.choice(k, p=p), rng.choice(k, p=q)
        probabilities.append((p.copy(), q.copy()))  # Used BEFORE this round's feedback.
        actions.append((a, b))
        regrets[0] += A[:, b] - A[a, b]
        regrets[1] += B[a, :] - B[a, b]  # Keep negative cumulative regrets.
    return np.asarray(actions), np.asarray(probabilities)

In [17]:
def show_rm(game="Matching pennies", rounds=1000, seed=1,
            bob_rule="Regret matching", bob_first=.5):
    g = GAMES[game]
    actions, probabilities = play_rm(g, rounds, seed, bob_rule, bob_first)
    data = history_data(g, actions)
    labels = g["labels"]
    fig, axes = plt.subplots(3, 3, figsize=(14, 11), layout="constrained")
    start = max(0, rounds-150)
    for i, name in enumerate(("Alice", "Bob")):
        for a, label in enumerate(labels):
            axes[0, i].plot(data["t"][start:], probabilities[start:, i, a],
                            color=plt.get_cmap("tab10")(a), label=label,
                            marker="." if rounds < 10 else None)
        axes[0, i].set(ylim=(-.03, 1.03), xlabel="Round t", ylabel="Probability",
                       title=f"{name}: sampling probabilities\n(last {rounds-start} rounds)")
        axes[0, i].legend(fontsize=8, ncols=3)
        frequency_plot(axes[1, i], data, labels, player=i)
        axes[0, 2].plot(data["t"], data["rho"][:, i], color=COLORS[i], label=name,
                        ls="--" if i else "-", marker="o" if rounds == 1 else None)
        average = data["payoffs"][:, i].cumsum() / data["t"]
        axes[2, 2].plot(data["t"], average, color=COLORS[i], label=name,
                        ls="--" if i else "-", marker="o" if rounds == 1 else None)
    axes[0, 2].set(xscale="log", xlabel="Rounds played, T", ylabel="Utility units per round",
                   title="Average external regret\n(maximum = empirical CCE violation)")
    axes[0, 2].set_yscale("symlog", linthresh=1e-3); axes[0, 2].set_ylim(bottom=0)
    axes[0, 2].legend(); axes[0, 2].grid(alpha=.2)
    error_plot(axes[1, 2], data)
    distribution_plots(axes[2, :2], data, labels)
    axes[2, 2].set(xscale="log", xlabel="Rounds played, T", ylabel="Utility per round",
                   title="Average realized payoff")
    axes[2, 2].legend(); axes[2, 2].grid(alpha=.2)
    fig.suptitle(f"{game} | Alice: regret matching; Bob: {bob_rule.lower()} | T = {rounds:,}\n"
                 f"Maximum average regret / CCE violation = {data['rho'][-1].max():.4f} | "
                 f"Nash error = {data['nash_error'][-1]:.4f}", fontsize=12)
    return fig


rm_controls = dashboard(show_rm, {
    "game": widgets.Dropdown(options=list(GAMES), value="Matching pennies", description="Game"),
    "rounds": slider(1000, 1, 5000, "Rounds"),
    "seed": slider(1, 0, 100, "Random seed"),
    "bob_rule": widgets.Dropdown(options=["Regret matching", "Fixed mixture"], description="Bob's rule"),
    "bob_first": widgets.FloatSlider(value=.5, min=0, max=1, step=.05,
                                     description="Fixed Bob: P(first)", continuous_update=False),
})

def enable_fixed_bob(change=None):
    rm_controls["bob_first"].disabled = rm_controls["bob_rule"].value != "Fixed mixture"

rm_controls["bob_rule"].observe(enable_fixed_bob, names="value")
enable_fixed_bob()